# htd SFT varied (pre-registration R2)

Trains SFT exactly like `sft_lr1e4_s1` (seed 1, same arguments and library versions) on pairs v3 varied: the chosen replies that repeated an earlier chosen reply were rewritten (`scripts/vary_chosen.py`). Everything else in the pairs is identical to pairs v2.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    exact = [p for p in hits if "datasets/rockerleo/htd-code" in str(p)]
    hits = exact or hits
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted. input: " + str(sorted(str(p) for p in Path('/kaggle/input').glob('*'))))
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
SRC = find_repo()
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("code root:", SRC)
for rel, marker in {"scripts/train.py": "set_seed(a.seed)", "src/htd/train_data.py": "def filter_pair_types"}.items():
    assert marker in (WORK / rel).read_text(encoding="utf-8"), f"stale code dataset: {rel} lacks {marker!r}"
print("code markers ok")

def find_dir(marker, parent_depth):
    hits = [p for p in Path("/kaggle/input").rglob(marker) if p.exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit(f"{marker} not mounted")
    p = hits[0]
    for _ in range(parent_depth):
        p = p.parent
    return p

def read_token():
    for p in Path("/kaggle/input").rglob("hf_token.txt"):
        return p.read_text().strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

if not os.environ.get("HF_TOKEN"):
    tok = read_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ.setdefault("HUGGINGFACE_API_KEY", tok)
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets (private) or add a Kaggle secret HF_TOKEN"

FIRE = "CohereLabs/tiny-aya-fire"
GLOBAL = "CohereLabs/tiny-aya-global"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def assert_adapter(name):
    p = WORK / "outputs" / name
    cfg = p / "adapter_config.json"
    weights = p / "adapter_model.safetensors"
    assert cfg.exists(), f"{name}: adapter_config.json missing"
    assert weights.exists() and weights.stat().st_size > 1_000_000, f"{name}: adapter weights missing or empty"
    print(f"{name}: adapter ok")


In [ ]:
pairs_src = find_dir("dpo_train.jsonl", 1)
dst = WORK / "data" / "generated" / "pairs"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(pairs_src, dst)
print("pairs:", sorted(p.name for p in dst.iterdir()))
n_sft = sum(1 for x in open(dst / "sft_train.jsonl") if x.strip())
assert n_sft == 858, f"expected 858 SFT rows (pairs v3 varied), got {n_sft}"
assert (dst / "vary_manifest.json").exists(), "attach rockerleo/htd-pairs-v3-varied, not pairs v2"
print(json.loads((dst / "vary_manifest.json").read_text()))

In [ ]:
# Same library versions as sft_lr1e4 (htd-train v6), so the seed is the only change.
run([sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.20.2", "trl==0.29.1", "peft==0.21.2", "bitsandbytes>=0.45", "accelerate>=1.0"])

In [ ]:
import importlib.metadata as md
versions = {p: md.version(p) for p in ("transformers", "trl", "peft", "bitsandbytes", "torch")}
print(versions)
assert (versions["transformers"], versions["trl"], versions["peft"]) == ("5.20.2", "0.29.1", "0.21.2"), versions

In [ ]:
# Same arguments as sft_lr1e4_s1; only the chosen text of the pairs differs (pre-registration R2).
run([sys.executable, "scripts/train.py", "--method", "sft", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/sft_varied_s1",
     "--epochs", "1", "--lr", "1e-4", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8", "--seed", "1"])
assert_adapter("sft_varied_s1")
meta = json.loads((WORK / "outputs" / "sft_varied_s1" / "train_meta.json").read_text())
assert meta["args"]["seed"] == 1 and meta["train"]["rows"] == 858, meta
print(sorted(p.name for p in (WORK / "outputs" / "sft_varied_s1").iterdir()))